# 🎯 SAM 3 Text-Prompted Pipeline (Option A)
### Zero-Shot Segmentation with Natural Language

---

**No Detection Model Required!**

SAM 3 can segment objects using only text prompts:
```
Image → SAM 3 ("segment all cracks") → Masks → Skeleton → Length
```

**Key Features:**
- ✅ Text-prompted segmentation
- ✅ Open vocabulary (works with any concept)
- ✅ No bounding box or point prompts needed

## 1️⃣ Install Dependencies

In [ ]:
# @title 🚀 Install SAM 3
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install SAM 3 (Official Meta Release)
!pip install -q git+https://github.com/facebookresearch/segment-anything-3.git

# Install utilities
!pip install -q opencv-python pandas numpy tqdm scikit-image

# Download SAM 3 checkpoint
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything_3/sam3_vit_h.pt

print("\n✅ SAM 3 Setup Complete")

## 2️⃣ Configuration

In [ ]:
# @title ⚙️ Configuration
import os

# @markdown ### 📁 Input
IMAGES_FOLDER = "/content/images/"  # @param {type:"string"}

# @markdown ### 🎯 Text Prompt for SAM 3
TEXT_PROMPT = "cracks on the road surface"  # @param {type:"string"}
# @markdown *Examples: "cracks", "potholes", "road damage", "linear fractures"*

# @markdown ### 📏 Calibration
PIXELS_PER_METER = 500.0  # @param {type:"number"}

# @markdown ### 🎛️ Detection Type
DEFECT_TYPE = "cracks"  # @param ["cracks", "potholes"]

print(f"✅ Config loaded")
print(f"   Text prompt: '{TEXT_PROMPT}'")
print(f"   Scale: {PIXELS_PER_METER} px/m")

## 3️⃣ Load SAM 3

In [ ]:
# @title 🎭 Load SAM 3 Model

from sam3 import build_sam3, SAM3Predictor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("🔄 Loading SAM 3...")
sam3_model = build_sam3(
    checkpoint="sam3_vit_h.pt",
    device=device
)
sam3_predictor = SAM3Predictor(sam3_model)

print("✅ SAM 3 Loaded")
print("   Supports: text prompts, visual prompts, box/point prompts")

## 4️⃣ Measurement Functions

In [ ]:
# @title 📏 Measurement Functions

import cv2
import numpy as np
from skimage.morphology import skeletonize

def measure_defect(mask, ppm, defect_type):
    """Measure defect from mask."""
    clean_mask = mask > 0.0
    
    if 'crack' in defect_type.lower():
        skeleton = skeletonize(clean_mask)
        pixel_length = np.sum(skeleton)
        
        if pixel_length == 0:
            contours, _ = cv2.findContours(
                clean_mask.astype(np.uint8) * 255,
                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            if contours:
                pixel_length = sum(cv2.arcLength(c, True) for c in contours) / 2
        
        length_m = pixel_length / ppm
        return max(length_m, 0.01) if pixel_length > 0 else 0.0, (skeleton, clean_mask)
    else:
        pixel_area = np.sum(clean_mask)
        area_m2 = pixel_area / (ppm ** 2)
        return max(area_m2, 0.001) if pixel_area > 0 else 0.0, clean_mask

print("✅ Measurement functions ready")

## 5️⃣ Run SAM 3 Text Pipeline

In [ ]:
# @title 🚀 Run SAM 3 Text-Prompted Pipeline

import glob
import pandas as pd
import shutil
from tqdm import tqdm

def run_sam3_text_pipeline():
    # Find images
    image_files = glob.glob(os.path.join(IMAGES_FOLDER, "*.jpg")) + \
                  glob.glob(os.path.join(IMAGES_FOLDER, "*.jpeg")) + \
                  glob.glob(os.path.join(IMAGES_FOLDER, "*.png"))
    
    if not image_files:
        print(f"❌ No images found in {IMAGES_FOLDER}")
        return pd.DataFrame()
    
    print(f"✅ Found {len(image_files)} images")
    print(f"🎯 Text Prompt: '{TEXT_PROMPT}'")
    
    os.makedirs("results/unmasked", exist_ok=True)
    os.makedirs("results/masked", exist_ok=True)
    
    all_records = []
    
    for img_path in tqdm(image_files, desc="SAM 3 Processing"):
        filename = os.path.basename(img_path)
        print(f"\n📸 {filename}")
        
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        
        height, width = frame.shape[:2]
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # SAM 3 Text-Prompted Prediction
        sam3_predictor.set_image(frame_rgb)
        
        try:
            # Text prompt segmentation (SAM 3 feature)
            masks, scores, boxes = sam3_predictor.predict_text(
                text_prompt=TEXT_PROMPT,
                return_boxes=True
            )
            
            print(f"   Found {len(masks)} segments")
            
            for idx, (mask, score, bbox) in enumerate(zip(masks, scores, boxes)):
                # Measure
                measurement, mask_data = measure_defect(mask, PIXELS_PER_METER, DEFECT_TYPE)
                
                if 'crack' in DEFECT_TYPE.lower():
                    label = f"{DEFECT_TYPE} | Length: {measurement:.2f}m"
                    measurement = min(measurement, 5.0)
                else:
                    label = f"{DEFECT_TYPE} | Area: {measurement:.2f}m²"
                    measurement = min(measurement, 2.0)
                
                out_filename = f"{DEFECT_TYPE}_{idx}_{filename}"
                x1, y1, x2, y2 = map(int, bbox)
                
                # Save UNMASKED
                unmasked = frame.copy()
                cv2.rectangle(unmasked, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(unmasked, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                cv2.imwrite(f"results/unmasked/{out_filename}", unmasked)
                
                # Save MASKED
                masked = frame.copy()
                overlay = np.zeros_like(masked)
                
                if 'crack' in DEFECT_TYPE.lower():
                    skeleton, clean_mask = mask_data
                    overlay[clean_mask] = (0, 200, 200)
                    overlay[skeleton] = (255, 255, 0)
                else:
                    clean_mask = mask_data
                    overlay[clean_mask] = (0, 0, 255)
                
                masked = cv2.addWeighted(masked, 0.7, overlay, 0.3, 0)
                cv2.rectangle(masked, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(masked, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                cv2.imwrite(f"results/masked/{out_filename}", masked)
                
                # Record
                record = {
                    'source_image': filename,
                    'detection_idx': idx,
                    'confidence': float(score),
                    'image_path': out_filename,
                    'text_prompt': TEXT_PROMPT
                }
                if 'crack' in DEFECT_TYPE.lower():
                    record['length_m'] = round(measurement, 4)
                else:
                    record['area_m2'] = round(measurement, 4)
                
                all_records.append(record)
                
        except Exception as e:
            print(f"   ⚠️ Error: {e}")
            continue
    
    # Save results
    print("\n" + "="*60)
    print("📊 SAM 3 TEXT PIPELINE COMPLETE")
    print("="*60)
    
    if all_records:
        df = pd.DataFrame(all_records)
        df.to_csv("results/detections.csv", index=False)
        print(f"✅ Detections: {len(df)}")
        
        shutil.make_archive("sam3_results", 'zip', "results")
        print(f"\n📦 sam3_results.zip created")
        
        try:
            from google.colab import files
            files.download("sam3_results.zip")
        except:
            pass
        
        return df
    else:
        print("⚠️ No detections found.")
        return pd.DataFrame()

df = run_sam3_text_pipeline()
df.head()